# Notebook 01: Data Loading & Baseline Verification

**Memory x RL Interaction Experiment**

This notebook:
1. Loads all 4 datasets and verifies sizes/format
2. Prints sample problems from each
3. Validates baseline accuracy assumptions
4. Tests answer parsing on sample outputs

In [ ]:
import sys
sys.path.insert(0, '..')

from lib.config import ExperimentConfig
from lib.data import load_math500_l4l5, load_aime_2025, load_olympmath_easy, load_amc12
from lib.answer_parsing import parse_answer, check_answer, strip_think_blocks

CFG = ExperimentConfig()
print(f"Model: {CFG.MODEL_NAME}")
print(f"Training epochs: {CFG.GRPO_EPOCHS}")
print(f"Seeds: {CFG.SEEDS}")

## 1. Load Training Data: MATH-500 L4-5

In [ ]:
math500 = load_math500_l4l5()
print(f"MATH-500 L4-5: {len(math500)} problems")
print(f"Expected: ~200 problems")
print(f"\nSample problem:")
print(f"  ID: {math500[0]['id']}")
print(f"  Problem: {math500[0]['problem'][:200]}...")
print(f"  Answer: {math500[0]['answer']}")

## 2. Load Eval Data: AIME 2025

In [ ]:
aime2025 = load_aime_2025()
print(f"AIME 2025: {len(aime2025)} problems")
print(f"Expected: 30 problems")
print(f"\nSample:")
for p in aime2025[:3]:
    print(f"  {p['id']}: answer={p['answer']}, problem={p['problem'][:100]}...")

## 3. Load Eval Data: OlymMATH-EASY

In [ ]:
olymp = load_olympmath_easy()
print(f"OlymMATH-EASY: {len(olymp)} problems")
print(f"Expected: ~100 problems")
if olymp:
    print(f"\nSample:")
    for p in olymp[:3]:
        print(f"  {p['id']}: answer={p['answer']}, problem={p['problem'][:100]}...")

## 4. Load Eval Data: AMC 12

In [ ]:
amc12 = load_amc12()
print(f"AMC 12: {len(amc12)} problems")
print(f"Expected: 50 problems (manual entry required)")
if amc12:
    print(f"\nSample:")
    for p in amc12[:3]:
        print(f"  {p['id']}: answer={p['answer']}")

## 5. Dataset Summary

In [ ]:
print("Dataset Summary")
print("=" * 50)
datasets = {
    "MATH-500 L4-5 (train)": math500,
    "AIME 2025 (eval)": aime2025,
    "OlymMATH-EASY (eval)": olymp,
    "AMC 12 (eval)": amc12,
}
total_eval = 0
for name, ds in datasets.items():
    print(f"  {name}: {len(ds)} problems")
    if "eval" in name:
        total_eval += len(ds)
print(f"\nTotal eval problems: {total_eval}")
print(f"Target: 180 problems")

## 6. Answer Parsing Tests

In [ ]:
# Test basic parsing
tests = [
    ("The answer is \\boxed{42}", "42"),
    ("#### 7", "7"),
    ("The answer is 100.", "100"),
    ("\\boxed{\\frac{1}{2}}", "\\frac{1}{2}"),
    ("<think>Let me think...\n42 is wrong</think>The answer is \\boxed{7}", "7"),
]

print("Answer Parsing Tests:")
all_pass = True
for raw, expected in tests:
    result = parse_answer(raw)
    status = "PASS" if result == expected else "FAIL"
    if status == "FAIL":
        all_pass = False
    print(f"  {status}: parse_answer({raw[:50]}...) = {result} (expected {expected})")

# Test check_answer
assert check_answer("42", "42") == True
assert check_answer("042", "42") == True
assert check_answer("7", "8") == False
print(f"\ncheck_answer tests: PASS")

# Test strip_think_blocks
text = "<think>Long thinking process...</think>The final answer is \\boxed{42}"
stripped = strip_think_blocks(text)
assert "<think>" not in stripped
assert "42" in stripped
print(f"strip_think_blocks test: PASS")

print(f"\nAll parsing tests: {'PASS' if all_pass else 'SOME FAILURES'}")

## 7. Baseline Pass@1 (Optional - requires GPU)

Run this cell on Colab with GPU to verify baseline accuracy assumptions.

In [ ]:
# Uncomment to run baseline on GPU (Colab)
# from vllm import LLM, SamplingParams
# from transformers import AutoTokenizer
#
# tokenizer = AutoTokenizer.from_pretrained(CFG.MODEL_NAME)
# llm = LLM(model=CFG.MODEL_NAME, dtype="bfloat16", gpu_memory_utilization=0.9)
#
# def format_prompt(problem):
#     messages = [
#         {"role": "system", "content": "Solve step by step. Put answer in \\boxed{}."},
#         {"role": "user", "content": problem},
#     ]
#     return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
#
# # Test on first 20 MATH-500 problems
# sample = math500[:20]
# prompts = [format_prompt(p["problem"]) for p in sample]
# params = SamplingParams(n=1, temperature=0.01, max_tokens=4096)
# outputs = llm.generate(prompts, params)
#
# correct = 0
# for p, o in zip(sample, outputs):
#     answer = parse_answer(o.outputs[0].text)
#     if check_answer(answer, p["answer"]):
#         correct += 1
# print(f"Baseline Pass@1 on 20 MATH-500 L4-5: {correct}/20 = {correct/20:.1%}")
# print(f"Expected: ~75-80%")